In [1]:
import os
import glob
import pandas as pd
import numpy as np
from xgboost import XGBClassifier as xgb
from sklearn.ensemble import RandomForestClassifier as rf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import subprocess
from tqdm import tqdm

In [10]:
BASE_DIR = r"E:\Research Project (Prof. Preeti Rao)\Top 100 files by Wheeze count"
OUTPUT_DIR = r"E:\Research Project (Prof. Preeti Rao)\Classification_24B3907\mfcc training data"
SMILE_PATH = r"E:\Research Project (Prof. Preeti Rao)\opensmile-3.0-win-x64\bin\SMILExtract.exe"
CONFIG_PATH = r"E:\Research Project (Prof. Preeti Rao)\opensmile-3.0-win-x64\myconfig\pitch_dir.conf"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# FIND ALL .wav/.txt PAIRS
wav_files = glob.glob(os.path.join(BASE_DIR, "*.wav"))
txt_files = [f.replace('.wav', '_label_audacity.txt') for f in wav_files]
valid_pairs = [(wav, txt) for wav, txt in zip(wav_files, txt_files) if os.path.exists(txt)]
print(f"Found {len(valid_pairs)} valid .wav/.txt pairs")

Found 100 valid .wav/.txt pairs


In [ ]:
#PROCESS 80% FOR TRAINING (FIRST 80 FILES) 
train_pairs = valid_pairs[:80]
test_pairs = valid_pairs[80:100]  # Last 20 for validation
all_train_dfs = []

print("Processing 80 training files...")
for wav_file, txt_file in tqdm(train_pairs, desc="Train files"):
    base_name = os.path.basename(wav_file).replace('.wav', '')
    output_csv = os.path.join(OUTPUT_DIR, f"{base_name}_energy.csv")
    
    # 1. Extract features with openSMILE
    cmd = [SMILE_PATH, "-C", CONFIG_PATH, "-I", wav_file, "-O", output_csv]
    subprocess.run(cmd, check=True, capture_output=True)
    
    # 2. Parse single-column CSV
    df = pd.read_csv(output_csv, header=None)
    df_split = df[0].str.split(";", expand=True)
    df_split.columns = [
    "frameIndex",
    "frameTime",
    "F0_Direction",
    "Direction_Score",
    "pcm_LOGenergy"
]
    
    # 3. Add wheeze labels from .txt
    label_df = pd.read_csv(txt_file, sep="\t", header=None, names=["start", "end", "label"])
    wheeze_intervals = label_df[label_df["label"] == "Wheeze"][["start", "end"]].values.tolist()
    
    df_split["frameTime"] = pd.to_numeric(df_split["frameTime"], errors='coerce')
    def is_wheeze(frame_times, intervals):
        labels = pd.Series(0, index=frame_times.index)
        for start, end in intervals:
            mask = (frame_times >= start) & (frame_times <= end)
            labels[mask] = 1
        return labels
    df_split["Wheeze"] = is_wheeze(df_split["frameTime"], wheeze_intervals)
    df_split = df_split[df_split["frameTime"].notna()].reset_index(drop=True)
    
    # Save processed file
    df_split.to_csv(output_csv, index=False)
    
    # Add file identifier
    df_split["file_id"] = base_name
    all_train_dfs.append(df_split)
    print(f"{base_name}: {len(df_split)} frames, {df_split['Wheeze'].sum()} wheezes")

Processing 80 training files...


Train files:   1%|▏         | 1/80 [00:00<00:13,  5.84it/s]

steth_20181001_11_01_50: 1497 frames, 667 wheezes


Train files:   2%|▎         | 2/80 [00:00<00:10,  7.33it/s]

steth_20181001_11_02_11: 1497 frames, 787 wheezes


...

trunc_2019-05-31-14-07-30-L5_13: 1497 frames, 1292 wheezes
trunc_2019-05-31-14-07-30-L5_4: 1497 frames, 933 wheezes


Train files:  99%|█████████▉| 79/80 [00:09<00:00,  6.53it/s]

trunc_2019-05-31-14-37-49-L1_13: 1497 frames, 1013 wheezes
trunc_2019-05-31-14-37-49-L1_14: 1497 frames, 801 wheezes


Train files: 100%|██████████| 80/80 [00:09<00:00,  8.16it/s]

trunc_2019-05-31-14-37-49-L2_13: 1497 frames, 1000 wheezes


In [7]:
# COMBINE ALL TRAINING DATA 
train_df = pd.concat(all_train_dfs, ignore_index=True)
print(f"\n TRAINING SET: {train_df.shape[0]:,} total frames across {len(train_pairs)} files")
print("Wheeze distribution:\n", train_df['Wheeze'].value_counts())

# Cast to numeric
num_cols = [
    "frameTime",
    "F0_Direction",
    "Direction_Score"
]
for c in num_cols:
    train_df[c] = pd.to_numeric(train_df[c], errors="coerce")

# Drop rows with missing key features
train_df = train_df.dropna(
    subset=[
    "frameTime",
    "F0_Direction",
    "Direction_Score"
]).reset_index(drop=True)


features = [
    "F0_Direction",
    "Direction_Score"
]

X_train = train_df[features]
y_train = train_df["Wheeze"]



 TRAINING SET: 119,760 total frames across 80 files
Wheeze distribution:
 Wheeze
1    66737
0    53023
Name: count, dtype: int64


In [8]:
print("\n Training...")

model = xgb(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=len(y_train[y_train==0]) / len(y_train[y_train==1]),
    random_state=42,
    eval_metric='auc',
)
model.fit(X_train, y_train, verbose=10)

# pos_weight = len(y_train[y_train==0]) / len(y_train[y_train==1])
# model = rf(
#     n_estimators=200,
#     max_depth=6,
#     max_samples=0.8,
#     max_features=0.8,
#     class_weight={0:1, 1:pos_weight},
#     random_state=42,
# )
# model.fit(X_train, y_train)

print("\n🔍 Processing test files...")
test_dfs = []

# UPDATED expected columns for pitch_energy CSV
expected_cols = [
    "frameIndex", "frameTime",
    "F0_Direction",
    "Direction_Score"
]

def parse_smile_csv(raw_csv_path, txt_path):
    """Read single-column SMILE CSV, split into 8 cols, add Wheeze, save wide CSV."""
    raw = pd.read_csv(raw_csv_path, header=None)
    df_split = raw[0].str.split(";", expand=True)
    df_split.columns = expected_cols  # <-- changed

    label_df = pd.read_csv(
        txt_path, sep="\t", header=None,
        names=["start", "end", "label"]
    )
    wheeze_intervals = (
        label_df[label_df["label"] == "Wheeze"][["start", "end"]]
        .values
        .tolist()
    )

    df_split["frameTime"] = pd.to_numeric(df_split["frameTime"], errors="coerce")
    df_split["Wheeze"] = is_wheeze(df_split["frameTime"], wheeze_intervals)
    df_split = df_split[df_split["frameTime"].notna()].reset_index(drop=True)

    df_split.to_csv(raw_csv_path, index=False)
    return df_split


for wav_file, txt_file in tqdm(test_pairs, desc="Test files"):
    base_name = os.path.basename(wav_file).replace(".wav", "")
    output_csv = os.path.join(OUTPUT_DIR, f"{base_name}_energy.csv")

    if not os.path.exists(output_csv):
        cmd = [SMILE_PATH, "-C", CONFIG_PATH, "-I", wav_file, "-O", output_csv]
        subprocess.run(cmd, check=True, capture_output=True)
        df_test = parse_smile_csv(output_csv, txt_file)
    else:
        df_test = pd.read_csv(output_csv)
        # if columns are not the wide format, regenerate and parse
        if not set(expected_cols).issubset(df_test.columns):
            cmd = [SMILE_PATH, "-C", CONFIG_PATH, "-I", wav_file, "-O", output_csv]
            subprocess.run(cmd, check=True, capture_output=True)
            df_test = parse_smile_csv(output_csv, txt_file)

    # UPDATED numeric casting for new feature set
    df_test["frameTime"]          = pd.to_numeric(df_test["frameTime"], errors="coerce")
    df_test["F0_Direction"]   = pd.to_numeric(df_test["F0_Direction"], errors="coerce")
    df_test["Direction_Score"] = pd.to_numeric(df_test["Direction_Score"], errors="coerce")

    # UPDATED dropna subset: only new features
    df_test = df_test.dropna(
        subset=[
            "frameTime",
            "F0_Direction",
            "Direction_Score"
        ]
    ).reset_index(drop=True)

    df_test["file_id"] = base_name
    test_dfs.append(df_test)

test_df = pd.concat(test_dfs, ignore_index=True)
X_test = test_df[features]
y_test = test_df["Wheeze"]


 Training...

🔍 Processing test files...


Test files: 100%|██████████| 20/20 [00:02<00:00,  9.28it/s]


In [9]:
y_pred = model.predict(X_test)
print("\n RESULTS ")
print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


 RESULTS 
              precision    recall  f1-score   support

           0       0.45      0.70      0.55     13503
           1       0.55      0.31      0.39     16437

    accuracy                           0.48     29940
   macro avg       0.50      0.50      0.47     29940
weighted avg       0.51      0.48      0.46     29940


Confusion Matrix:
[[ 9418  4085]
 [11390  5047]]
